# DQVP End-to-End Reproducibility Notebook

Reproduces the pipeline described in the paper *Towards a Minimal ODRL Profile for Data Quality Validation in Data Spaces Governance* (W3C ODRL Workshop 2026).

Three Data Quality Requirements, each illustrating one structural ODRL barrier:

| DQR | DQVP leftOperand | Barrier | Check |
|-----|------------------|---------|-------|
| DQR1EH | `dqvp:NonNullRate` | B1 | `nationalHealthcarePatientID` is non-null in at least 99% of records |
| DQR2EH | `dqvp:ReferenceConformance` | B1, B2 | `countryOfAffiliation` conforms to ISO 3166 |
| DQR3EH | `dqvp:NullRate` | B1, B3 | where `gender = male`, `pregnancyHistory` null rate is 100% |


In [ ]:
import os, sys, json, importlib.util
from pathlib import Path

# Add ~/.local/bin so the Jupyter kernel finds the EYE reasoner
os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ.get("PATH", "")

HERE = Path().resolve()   # .../PositionPaper/reproduction/

import shutil
from dqvp_pipeline import build_dqvp_turtle, build_combined_dqvp_turtle, plan_dqvp, generate_gx_code
from rdflib import Graph, Namespace, RDF

TB   = Namespace("http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#")
AB   = Namespace("http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#")
ODRL = Namespace("http://www.w3.org/ns/odrl/2/")
DQVP = Namespace("https://github.com/feed-upc/DQVP-position/blob/main/dq-profile.ttl#")

print(f"EYE: {shutil.which('eye')}")
print("ready")


In [ ]:
# DQR1EH  NonNullRate          resolves B1          expects: FAIL (10 injected nulls, 98% < 99%)
# DQR2EH  ReferenceConformance resolves B1 + B2     expects: FAIL (10 bad country codes)
# DQR3EH  NullRate             resolves B1 + B3     expects: PASS

DQR_ID   = "DQR1EH"
DATA_CSV = HERE / "data" / "Patient_Summary.csv"

req = json.loads((HERE / "requirements" / f"{DQR_ID}.json").read_text())
print(f"DQR:     {req['id']}")
print(f"Pattern: {req['pattern']['id']}")
print(f"Statement: {req['statement']}")


## Step 1: `build_dqvp_turtle()`

DQR JSON to DQVP policy. The DQR pattern maps to a `dqvp:QualityPolicy` whose `odrl:Duty` carries each metric as an `odrl:constraint` typed `dqvp:DQConstraint`.

In [ ]:
ttl = build_dqvp_turtle(req)
print(ttl)

In [ ]:
# Parse and query the Turtle to confirm structure
policy_graph = Graph()
policy_graph.parse(data=ttl, format="turtle")

print(f"Triples              : {len(policy_graph)}")
print(f"QualityPolicy        : {[str(p).split('/')[-1] for p in policy_graph.subjects(RDF.type, DQVP.QualityPolicy)]}")
print(f"leftOperand          : {[str(o).split('#')[-1] for _,_,o in policy_graph.triples((None, ODRL.leftOperand, None))]}")
print(f"targetAttribute      : {[str(o).split('#')[-1] for _,_,o in policy_graph.triples((None, DQVP.targetAttribute, None))]}")
print(f"referenceStandard    : {[str(o) for _,_,o in policy_graph.triples((None, DQVP.referenceStandard, None))]}")
print(f"conditionAttribute   : {[str(o).split('#')[-1] for _,_,o in policy_graph.triples((None, DQVP.conditionAttribute, None))]}")
print(f"conditionValues      : {[str(o) for _,_,o in policy_graph.triples((None, DQVP.conditionValues, None))]}")
print(f"rightOperand         : {[str(o) for _,_,o in policy_graph.triples((None, ODRL.rightOperand, None))]}")

## Step 2: `plan_dqvp()`

Turtle policy to plan graph. The planner bootstraps a `tb:PolicyChecker` plus `tb:LoadData` for each `dqvp:QualityPolicy`, then reasons over the standard `odrl:constraint` triples with one EYE N3 rule per supported `dqvp:leftOperand` term to produce a fan-out DAG of operation nodes.

In [ ]:
import tempfile

ttl_file = Path(tempfile.mkdtemp()) / f"{DQR_ID}_dqvp.ttl"
ttl_file.write_text(ttl)

plan = plan_dqvp(
    ttl_path   = str(ttl_file),
    data_path  = str(DATA_CSV),
    rules_dir  = HERE / "rules_n3_dqvp",
)

print(f"Plan triples: {len(plan)}")

In [ ]:
print("=== PolicyChecker ===")
for pc in plan.subjects(RDF.type, TB.PolicyChecker):
    policy   = plan.value(pc, TB.accordingTo)
    load_op  = plan.value(pc, TB.nextStep)
    branches = list(plan.objects(load_op, TB.nextStep)) if load_op else []
    print(f"  URI         : {str(pc).split('#')[-1]}")
    print(f"  accordingTo : {str(policy).split('/')[-1]}")
    print(f"  branches    : {len(branches)} check(s) from LoadData")

print("\n=== Operations ===")
for op in plan.subjects(RDF.type, TB.Operation):
    abstract = plan.value(op, TB.hasAbstract)
    terminal = plan.value(op, TB.isTerminal)
    inputs   = [
        (str(plan.value(i, TB.role)), str(plan.value(i, TB.value)).split('#')[-1].split('/')[-1])
        for i in plan.objects(op, TB.hasInput)
    ]
    print(f"  {str(abstract).split('#')[-1]:28s}  {'[terminal]' if terminal else '          '}  inputs={inputs}")


In [ ]:
# Serialize the plan so you can open it in the IDE
plan_file = ttl_file.parent / f"{DQR_ID}_plan.ttl"
plan.bind("tb", TB); plan.bind("ab", AB); plan.bind("odrl", ODRL)
plan.serialize(destination=str(plan_file), format="turtle")
print(f"Plan graph: {plan_file}")
print("Search 'nextStep' to trace the operation chain.")

## Step 3: `generate_gx_code()`

Plan graph to validation service. Each `tb:hasAbstract` operation maps to a Great Expectations expectation, guided by `code_metadata.json`.

In [ ]:
gx_code = generate_gx_code(plan, HERE / "code_metadata.json")
print(gx_code)

## Step 4: Run the validation against the EHDS sample

| DQR | Expected | Reason |
|-----|----------|--------|
| DQR1EH | FAIL | 10 null `nationalHealthcarePatientID` values injected (98% < 99%) |
| DQR2EH | FAIL | 10 invalid country codes injected |
| DQR3EH | PASS | `pregnancyHistory` is 100% null where `gender = male` |


In [ ]:
import importlib.util as ilu

job_file = ttl_file.parent / f"{DQR_ID}_validation_job.py"
job_file.write_text(
    gx_code
    + "\n\nif __name__ == '__main__':\n"
    "    import sys\n"
    f"    print('Result:', run_gx_validation(sys.argv[1] if len(sys.argv) > 1 else r'{DATA_CSV}'))\n"
)

spec = ilu.spec_from_file_location("job", job_file)
job  = ilu.module_from_spec(spec)
spec.loader.exec_module(job)

result = job.run_gx_validation(str(DATA_CSV))

print(f"\n{'='*50}")
print(f"  DQR    : {DQR_ID}")
print(f"  Data   : {DATA_CSV.name}")
print(f"  Result : {'PASS' if result else 'FAIL'}")
print(f"{'='*50}")


## Run all three DQRs in one sweep

In [ ]:
import importlib.util as ilu, tempfile

EXPECTED = {"DQR1EH": False, "DQR2EH": False, "DQR3EH": True}

for dqr_id, expected in EXPECTED.items():
    req_  = json.loads((HERE / "requirements" / f"{dqr_id}.json").read_text())
    ttl_  = build_dqvp_turtle(req_)

    tmp = Path(tempfile.mkdtemp())
    tf  = tmp / f"{dqr_id}.ttl"
    tf.write_text(ttl_)

    plan_ = plan_dqvp(str(tf), str(DATA_CSV), rules_dir=HERE / "rules_n3_dqvp")
    code_ = generate_gx_code(plan_, HERE / "code_metadata.json")

    jf = tmp / f"{dqr_id}_job.py"
    jf.write_text(code_)
    spec = ilu.spec_from_file_location("j", jf)
    mod  = ilu.module_from_spec(spec)
    spec.loader.exec_module(mod)

    result = mod.run_gx_validation(str(DATA_CSV))
    ok     = result is expected
    status = 'PASS' if result else 'FAIL'
    print(f"  {dqr_id}  {status}  {'as expected' if ok else 'unexpected'}")


## Combined policy

All three DQRs in a single `dqvp:QualityPolicy` typed as `odrl:Offer`, attached to the same `odrl:Duty` as conjunctive `odrl:constraint` triples per ODRL 2.2 semantics. This matches the listing in Figure 1 of the paper.

In [ ]:
all_reqs = [
    json.loads((HERE / "requirements" / "DQR1EH.json").read_text()),
    json.loads((HERE / "requirements" / "DQR2EH.json").read_text()),
    json.loads((HERE / "requirements" / "DQR3EH.json").read_text()),
]

combined_ttl = build_combined_dqvp_turtle(all_reqs, policy_id="ps-dq-offer-01")
print(combined_ttl)